In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, TimestampType
)

SOURCE_PATH = "/Volumes/workspace/gagealspach/ams_demo_staging/data"

source_schema = StructType([
    StructField("shipment_id", StringType(), True),
    StructField("status", StringType(), True),
    StructField("location", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("_source_version", LongType(), True),
    StructField("_operation", StringType(), True),
    StructField("_source_updated_ts", TimestampType(), True),
])

@dp.table(
    name="shipments_bronze",
    comment="Append-only shipment CDC events from Auto Loader"
)
def shipments_bronze():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_bronze_rescued_data")
        .schema(source_schema)
        .load(SOURCE_PATH)
        .withColumn("_source_file", F.col("_metadata.file_name"))
        .withColumn("_ingest_ts", F.current_timestamp())
    )